In [ ]:
!pip install tensorflow
!pip install tensorflow_datasets

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
# import seaborn as sns
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import matplotlib.image as mpimg
# import itertools

print(tf.__version__)

2.19.0


In [3]:
cifar10 = tf.keras.datasets.cifar10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 14s 0us/step


In [4]:
# Shuffle the training data first
indices = tf.random.shuffle(tf.range(len(x_train)))
x_train_shuffled = tf.gather(x_train, indices)
y_train_shuffled = tf.gather(y_train, indices)

# Split into training and validation (80/20)
val_split = 0.2
num_train = int(len(x_train_shuffled) * (1 - val_split))
x_train_final = x_train_shuffled[:num_train]
y_train_final = y_train_shuffled[:num_train]
x_val = x_train_shuffled[num_train:]
y_val = y_train_shuffled[num_train:]

print(f"Training set size: {len(x_train_final)}")
print(f"Validation set size: {len(x_val)}")
print(f"Test set size: {len(x_test)}")

Training set size: 40000
Validation set size: 10000
Test set size: 10000


In [5]:
dataset, info = tfds.load("cifar10", as_supervised=True, with_info=True)
dataset_size = info.splits["train"].num_examples # 3670
class_names = info.features["label"].names # ["dandelion", "daisy", ...]
n_classes = info.features["label"].num_classes # 5

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.QQT0W2_3.0.2/cifar10-train.tfrecord*...:   0%|         …

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/cifar10/incomplete.QQT0W2_3.0.2/cifar10-test.tfrecord*...:   0%|          …

Dataset cifar10 downloaded and prepared to /root/tensorflow_datasets/cifar10/3.0.2. Subsequent calls will reuse this data.


In [6]:
sample_images = (x_train[:4].astype('float32')) / 255.0  # Normalize to 0-1
# Resize to 299x299 for Xception
images_resized = tf.image.resize(sample_images, (299, 299))

In [7]:
inputs = tf.keras.applications.xception.preprocess_input(images_resized)

In [8]:
batch_size = 32
preprocess = tf.keras.Sequential([
tf.keras.layers.Resizing(height=299, width=299, crop_to_aspect_ratio=True),
tf.keras.layers.Lambda(tf.keras.applications.xception.preprocess_input)
])

train_set = tf.data.Dataset.from_tensor_slices((x_train_final, y_train_final)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
valid_set = tf.data.Dataset.from_tensor_slices((x_val, y_val)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)
test_set = tf.data.Dataset.from_tensor_slices((x_test, y_test)).map(lambda X, y: (preprocess(X), y)).batch(batch_size)

In [9]:
data_augmentation = tf.keras.Sequential([
tf.keras.layers.RandomFlip(mode="horizontal", seed=42),
tf.keras.layers.RandomRotation(factor=0.05, seed=42),
tf.keras.layers.RandomContrast(factor=0.2, seed=42)
])


In [ ]:
# instantiate the base model
base_model = tf.keras.applications.xception.Xception(weights="imagenet", include_top=False)

# define the input shape (Xception expects 299x299 images)
inputs = tf.keras.Input(shape=(299, 299, 3))

# add the data augmentation as the first layer
x = data_augmentation(inputs)

# pass the augmented images through to the base model, note: training=False so BatchNormalization layers don't get messed up
x = base_model(x, training=False)

# add rest of the custom head
x = tf.keras.GlobalAveragePooling2D()(x)
outputs = tf.keras.layers.Dense(n_classes, activation="softmax")(x)

# create the final model
Model = tf.keras.Model(inputs=inputs, outputs=outputs)

# Freeze the base model layers
for layer in base_model.layers:
    layer.trainable = False



83683744/83683744 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step


In [11]:
optimizer = tf.keras.optimizers.SGD(learning_rate=0.1, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = Model.fit(train_set, validation_data=valid_set, epochs=3)

Epoch 1/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 464s 339ms/step - accuracy: 0.8406 - loss: 0.5604 - val_accuracy: 0.8702 - val_loss: 0.4629
Epoch 2/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 384s 307ms/step - accuracy: 0.8725 - loss: 0.4473 - val_accuracy: 0.8814 - val_loss: 0.4271
Epoch 3/3
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 383s 307ms/step - accuracy: 0.8871 - loss: 0.3848 - val_accuracy: 0.8809 - val_loss: 0.4296


In [12]:
for layer in base_model.layers[56:]:
    layer.trainable = True

In [13]:
optimizer = tf.keras.optimizers.SGD(learning_rate=1e-4, momentum=0.9)
Model.compile(loss="sparse_categorical_crossentropy", optimizer=optimizer, metrics=["accuracy"])
history = Model.fit(train_set, validation_data=valid_set, epochs=10)

Epoch 1/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 789s 603ms/step - accuracy: 0.8803 - loss: 0.3974 - val_accuracy: 0.9018 - val_loss: 0.3183
Epoch 2/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 727s 582ms/step - accuracy: 0.9375 - loss: 0.1928 - val_accuracy: 0.9138 - val_loss: 0.2888
Epoch 3/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 728s 582ms/step - accuracy: 0.9589 - loss: 0.1320 - val_accuracy: 0.9187 - val_loss: 0.2800
Epoch 4/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 727s 582ms/step - accuracy: 0.9739 - loss: 0.0936 - val_accuracy: 0.9221 - val_loss: 0.2769
Epoch 5/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 728s 583ms/step - accuracy: 0.9839 - loss: 0.0677 - val_accuracy: 0.9241 - val_loss: 0.2762
Epoch 6/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 728s 582ms/step - accuracy: 0.9904 - loss: 0.0500 - val_accuracy: 0.9259 - val_loss: 0.2765
Epoch 7/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 729s 583ms/step - accuracy: 0.9940 - loss: 0.0377 - val_accuracy: 0.9278 - val_loss: 0.2773
Epoch 8/10
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 729s 583ms/step - ac